# Equilibrium constants

The equilibrium constants of the eight reversible reactions of the model (PGI to ENO), from the standard Gibbs energies of reaction of the component contribution method (`equilibrator_api`) at physiological conditions: pH 7.0, pMg 3.0, ionic strength 0.1 M and 37 C.

$$K_\mathrm{eq} = \exp\left(-\frac{\Delta_r G'^m}{R\,T}\right)$$

The constants are saved in `Data/k_eq_values.pkl`, which the model reads (`model/stoichiometry.load_keq`).

`equilibrator_api` is not part of the `hybridkin` environment: it downloads a large compound database. By default (`HK_RUN=0`) the notebook only loads and shows the saved constants. With `HK_RUN=1`, in an environment that has `equilibrator_api`, it recomputes them and overwrites `Data/k_eq_values.pkl`.

In [1]:
import os
import pickle

import numpy as np
import pandas as pd

HERE = os.getcwd()
REPO = os.path.dirname(HERE) if os.path.basename(HERE) == "notebooks" else HERE
KEQ_FILE = os.path.join(REPO, "Data", "k_eq_values.pkl")
RUN = os.environ.get("HK_RUN", "0") == "1"
print("KEQ_FILE :", KEQ_FILE)
print("RUN      :", RUN, "(False = load the saved constants; True = recompute them with equilibrator_api)")

KEQ_FILE : /Users/gabbi/Desktop/Repos/HybridKinetics-IIQ3733/Data/k_eq_values.pkl
RUN      : False (False = load the saved constants; True = recompute them with equilibrator_api)


## Computation

Each reaction is written with BiGG metabolite identifiers; `physiological_dg_prime` gives its Gibbs energy at 1 mM of every reactant (the physiological standard state).

In [2]:
if RUN:
    from equilibrator_api import ComponentContribution, Q_

    R = 8.31446261815324  # J/(mol K)
    T = 37.0 + 273.15  # K

    cc = ComponentContribution()
    cc.p_h = Q_(7.0)
    cc.p_mg = Q_(3.0)
    cc.ionic_strength = Q_(0.1, "M")
    cc.temperature = Q_(T, "K")

    reactions = {
        "pgi": "bigg.metabolite:g6p = bigg.metabolite:f6p",
        "pfk": "bigg.metabolite:f6p + bigg.metabolite:atp = bigg.metabolite:fdp + bigg.metabolite:adp",
        "fba": "bigg.metabolite:fdp = bigg.metabolite:g3p + bigg.metabolite:dhap",
        "tpi": "bigg.metabolite:dhap = bigg.metabolite:g3p",
        "gap": "bigg.metabolite:g3p + bigg.metabolite:nad + bigg.metabolite:pi = bigg.metabolite:13dpg + bigg.metabolite:nadh + bigg.metabolite:h",
        "pgk": "bigg.metabolite:13dpg + bigg.metabolite:adp = bigg.metabolite:3pg + bigg.metabolite:atp",
        "gpm": "bigg.metabolite:3pg = bigg.metabolite:2pg",
        "eno": "bigg.metabolite:2pg = bigg.metabolite:pep + bigg.metabolite:h2o",
    }

    k_eq = {}
    for name, formula in reactions.items():
        delta_g = cc.physiological_dg_prime(cc.parse_reaction_formula(formula))
        k_eq[name] = np.exp(-delta_g.value.m_as("J/mol") / (R * T))

    with open(KEQ_FILE, "wb") as fh:
        pickle.dump(k_eq, fh)
    print("written:", KEQ_FILE)

## Saved constants

In [3]:
with open(KEQ_FILE, "rb") as fh:
    k_eq = pickle.load(fh)

pd.Series({name: float(value) for name, value in k_eq.items()}, name="K_eq").rename_axis("reaction").to_frame()

,K_eq
reaction,
pgi,0.381019
pfk,341.297999
fba,0.161610
tpi,0.113015
gap,0.000260
pgk,1849.572177
gpm,0.187466
eno,4.527342
